## Lesson 4: Per-Residue Prediction (Token Classification)
### What you'll learn
- The difference between SEQUENCE-level prediction (one label per protein) and
  RESIDUE-level prediction (one label per amino acid).
- Use AutoModelForTokenClassification — same as NER in NLP.
- The trickiest part: ALIGNING labels to tokens (handling <cls>, <eos>, padding).

### The example task: 3-state secondary structure (Q3)
Each residue is labelled:
   H = alpha helix
   E = beta sheet
   C = coil / random coil

   Input:  "MKTVRQERLKSIVRILERSKEPVSGA..."
   Output: "CCCHHHHHHHHHHHHEEEEECCCCCC..."

Structurally identical to NER: each token gets a label, padding tokens get
the special label `-100` which the loss function ignores.

### About the dataset
SS3 datasets exist on Hugging Face under names like:
   "proteinea/secondary_structure"
   "El-Husseini/Protein_secondary_structure"
   FLIP / TAPE benchmarks
Column names vary. This script prints `dataset.features` so you can adjust
SEQUENCE_KEY / LABEL_KEY below to match. If the dataset doesn't exist or
columns differ, see the README for a list of working alternatives.

> **Run order matters.** The cells below build on each other. Run them **top to bottom** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit `NameError: name 'torch' is not defined` (or similar), you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports every library and defines the module-level constants the rest of the notebook relies on.

In [1]:
import os
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
DATASET_NAME = "agemagician/NetSurfP-SS3"
SEQUENCE_KEY = "input"
LABEL_KEY = "label"
LABEL_NAMES = ["H", "E", "C"]
OUTPUT_DIR = "./results/lesson4"
N_TRAIN = 200
N_VAL = 50
label_to_id = {l: i for i, l in enumerate(LABEL_NAMES)}
EPOCHS = 5


In [ ]:
# MLflow tracking -> shared local SQLite store.
# (repo is pip-installed via `pip install -e .`, so this import needs no sys.path shim)
import mlflow_utils as mu
mlflow = mu.setup_mlflow()

### `preprocess_example` (function)

Tokenize one sequence and align per-residue labels to tokens.

Why alignment is non-trivial:
- The tokenizer adds <cls> at position 0 and <eos> at the end.
- These have no matching amino acid in the input, so we set their
  labels to -100. The cross-entropy loss IGNORES positions with label -100.
- If the sequence is truncated to max_length, we truncate labels to match.

In [3]:
def preprocess_example(example, tokenizer):
    seq = example[SEQUENCE_KEY]
    ss = example[LABEL_KEY]  # e.g. "CCCHHHEEE..."

    tokenized = tokenizer(seq, truncation=True, max_length=512)
    n_tokens = len(tokenized["input_ids"])

    # The first token is <cls> — set label to -100 (ignore).
    aligned = [-100]

    # The middle tokens (n_tokens - 2 of them, if not truncated) correspond
    # 1:1 to amino acids in the input. Map them to label IDs.
    n_residue_tokens = n_tokens - 2  # subtract <cls> and <eos>
    for i in range(n_residue_tokens):
        if i < len(ss):
            aligned.append(label_to_id.get(ss[i], -100))
        else:
            aligned.append(-100)

    # Final <eos> token — also -100.
    aligned.append(-100)

    # Defensive: pad/truncate to exactly match input_ids length.
    aligned = aligned[:n_tokens] + [-100] * (n_tokens - len(aligned))
    tokenized["labels"] = aligned
    return tokenized

### `compute_metrics` (function)

Per-residue accuracy, ignoring positions with label -100.

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    mask = labels != -100  # ignore <cls>/<eos>/padding
    correct = (preds[mask] == labels[mask]).sum()
    total = mask.sum()
    return {"accuracy": float(correct) / float(total)}

### `main` (function)

In [5]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABEL_NAMES)
    )

    print(f"Loading dataset: {DATASET_NAME}")
    raw = load_dataset(DATASET_NAME)
    print(f"\nDataset features: {raw[list(raw.keys())[0]].features}")
    print(f"Splits available: {list(raw.keys())}")
    print(
        "\nIf SEQUENCE_KEY / LABEL_KEY at the top of this file don't match the "
        "feature names above, edit them and re-run.\n"
    )

    # Pick splits — datasets vary in their split names.
    train_split = "train" if "train" in raw else list(raw.keys())[0]
    val_split = "validation" if "validation" in raw else (
        "test" if "test" in raw else train_split
    )
    raw_train = raw[train_split].select(range(min(N_TRAIN, len(raw[train_split]))))
    raw_val = raw[val_split].select(range(min(N_VAL, len(raw[val_split]))))

    print(f"Train: {len(raw_train)} sequences, val: {len(raw_val)} sequences")

    # Preprocess: tokenize + align labels.
    train_ds = raw_train.map(
        lambda ex: preprocess_example(ex, tokenizer),
        remove_columns=raw_train.column_names,
    )
    val_ds = raw_val.map(
        lambda ex: preprocess_example(ex, tokenizer),
        remove_columns=raw_val.column_names,
    )

    # Hyperparameters logged to MLflow and reused by the Trainer below.
    LR, BATCH, WD = 2e-5, 4, 0.01
    run_name = "l4_esm2_8M_ss3"
    params = {
        "model": MODEL_NAME, "dataset": DATASET_NAME,
        "task": "Q3_secondary_structure", "n_train": len(raw_train),
        "n_val": len(raw_val), "epochs": EPOCHS, "lr": LR,
        "batch_size": BATCH, "weight_decay": WD,
    }

    # Everything from here on runs inside one MLflow run; the Trainer's
    # report_to="mlflow" streams per-step loss / per-epoch eval accuracy into
    # it, and we add the final eval metrics under clean names by hand.
    with mu.run("plm-secondary-structure", run_name, params=params,
                tags={"lesson": "plm_l4", "task": "token_classification"}):

        args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            fp16=torch.cuda.is_available(),  # mixed precision on GPU; no-op on CPU
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=LR,
            per_device_train_batch_size=BATCH,
            per_device_eval_batch_size=BATCH,
            num_train_epochs=EPOCHS,
            weight_decay=WD,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            greater_is_better=True,
            save_total_limit=2,
            logging_steps=20,
            report_to="mlflow",             # log loss/eval metrics into the active run
            run_name=run_name,
        )

        # DataCollatorForTokenClassification pads BOTH input_ids AND labels.
        # The label_pad_token_id=-100 ensures padded positions are ignored by loss.
        collator = DataCollatorForTokenClassification(
            tokenizer=tokenizer, label_pad_token_id=-100
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            processing_class=tokenizer,
            data_collator=collator,
            compute_metrics=compute_metrics,
        )

        print("\nTraining...")
        trainer.train()

        print("\nFinal evaluation:")
        metrics = trainer.evaluate()
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")

        # Clean metric names so per-residue accuracy lines up across lessons.
        mlflow.log_metrics({
            f"test_{k.replace('eval_', '')}": float(v)
            for k, v in metrics.items() if isinstance(v, (int, float))
        })

    # Inference on a held-out VALIDATION sequence, so we can line the model's
    # prediction up against the ground-truth secondary structure (actual SS3).
    sample = raw_val[0]
    seq = sample[SEQUENCE_KEY][:510]            # keep within the model's window
    true_ss = sample[LABEL_KEY][: len(seq)]
    inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=512).to(
        model.device
    )
    with torch.no_grad():
        logits = model(**inputs).logits[0]      # (L+2, num_labels)
    pred_ids = logits.argmax(dim=-1)[1 : 1 + len(seq)]   # drop <cls>, keep residues
    pred_ss = "".join(LABEL_NAMES[i] for i in pred_ids.tolist())
    agree = "".join("|" if p == t else " " for p, t in zip(pred_ss, true_ss))
    q3 = sum(p == t for p, t in zip(pred_ss, true_ss)) / max(1, len(true_ss))
    print(f"\nSequence:       {seq}")
    print(f"Predicted SS3:  {pred_ss}")
    print(f"Actual SS3:     {true_ss}")
    print(f"Agreement:      {agree}   ('|' = match)")
    print(f"Per-residue Q3 accuracy on this sequence: {q3:.3f}")

    print(
        """
Things to experiment with:
- 8-state SS (Q8) instead of Q3: see the "Getting the Q8 data" example cell below —
  set LABEL_NAMES to the 8 DSSP states and point SEQUENCE_KEY/LABEL_KEY at that dataset.
- Predict disorder regions (binary token classification — IDR vs structured).
- Predict binding sites — small datasets exist for this on Hugging Face.
- Switch model: "facebook/esm2_t12_35M_UR50D" generally improves SS accuracy noticeably.
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [6]:
main()

Using device: cuda
Loading model: facebook/esm2_t6_8M_UR50D


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmForTokenClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading dataset: agemagician/NetSurfP-SS3



Dataset features: {'input': Value('string'), 'label': Value('string'), 'disorder': Value('string'), 'dataset': Value('string')}
Splits available: ['train', 'validation', 'test']

If SEQUENCE_KEY / LABEL_KEY at the top of this file don't match the feature names above, edit them and re-run.

Train: 200 sequences, val: 50 sequences


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final evaluation:


Training Loss,Validation Loss,Epoch,Accuracy
0.685027,0.706907,5,0.719667


  eval_loss: 0.7069
  eval_accuracy: 0.7197



Sequence:       SLRFTASTSTPKSGSKIAKRGKKHPEPVASWMSEQRWAGEPEVMCTLQHKSIAQEAYKNYTITTSAVCKLVRQLQQQALSLQVHFERSERVLSGLQASSLPEALAGATQLLSHLDDFTATLERRGVFFNDAKIERRRYEQHLEQIRTVSKDTRYSLERQHYINLESLLDDVQLLKRHTLITLRLIFERLVRVLVISIEQSQCDLLLRANINMVATLMNIDYDGFRSLSDAFVQNEAVRTLLVVVLDHKQSSVRALALRALATLCCAPQAINQLGSCGGIEIVRDILQVESAGERGAIERREAVSLLAQITAAWHGSEHRVPGLRDCAESLVAGLAALLQPE
Predicted SS3:  CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHHHHHHHHHHHCCCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCCCHHHHHHHHHHHHHHHHHHHHHHHCCCCCCCCHHHHHHHHHHHHHHHHHHHHCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCCHHHHHHHHHHHHHHHHHHCCHHHHHHHHHHHHHHHHHHHHHHHHHHHCCHHHHHHHHHHHHHHHHCHHHHHHHHHCHHHHHHHHHHHHHCCCCCCCCHHHHHHHHHHHHHHHHHCCCCCCCCHHHHHHHHHHHHHHHHCCC
Actual SS3:     CCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHHHHHHHCCCCCCCCCCCCCCHHHHHHHHCCCCCCHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCCCCCHHHHHHHHHHHHHHHHHHHHHHHCCCCCCCCCHHHHHHHHHHHHCCCCCCCCCCCCCCCCCCCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCCHHHHHHHHHHHHHHCCCCCCCCCCCHHHHHHCCHHHHHHHHHHHCCCHHHHHHHHHHHHHHCCCH

## Example: getting the 8-state (Q8) data

The lesson above predicts **Q3** (H/E/C). The richer **Q8** scheme splits those into the 8
DSSP states — `H` α-helix, `G` 3₁₀-helix, `I` π-helix, `E` β-strand, `B` β-bridge, `T` turn,
`S` bend, `C` coil. Q8 is harder but more structurally informative.

`agemagician/NetSurfP-SS3` only ships Q3 labels, so for Q8 we load a dataset that carries an
`sst8` column. The cell below fetches one, prints its schema, the distinct 8-state labels and
their frequencies, and an example — and tells you exactly which Setup-cell constants to
change to train on Q8.

In [7]:
# Fetch a Q8 (8-state DSSP) secondary-structure dataset and inspect it.
# agemagician/NetSurfP-SS3 only carries Q3 labels, so we use one with an `sst8` column.
Q8_DATASET = "lamm-mit/protein-secondary-structure-nppe2"
Q8_SEQ_KEY, Q8_LABEL_KEY = "seq", "sst8"

q8 = load_dataset(Q8_DATASET)
split = "train" if "train" in q8 else list(q8.keys())[0]
print(f"Loaded {Q8_DATASET}")
print(f"  splits:   {list(q8.keys())}")
print(f"  features: {q8[split].features}")

# Derive the 8-state alphabet (and its frequencies) from a sample of the data.
from collections import Counter
counts = Counter()
for ex in q8[split].select(range(min(500, len(q8[split])))):
    counts.update(ex[Q8_LABEL_KEY])
q8_labels = sorted(counts)
print(f"\n  distinct Q8 labels ({len(q8_labels)}): {q8_labels}")
print(f"  label counts (first 500 seqs):   {dict(counts)}")

ex0 = q8[split][0]
print(f"\nExample sequence (first 80 aa): {ex0[Q8_SEQ_KEY][:80]}")
print(f"Example Q8 labels  (first 80):  {ex0[Q8_LABEL_KEY][:80]}")

print(
    "\nTo TRAIN on Q8 instead of Q3, set these in the Setup cell and re-run main():\n"
    f"  DATASET_NAME = \"{Q8_DATASET}\"\n"
    f"  SEQUENCE_KEY = \"{Q8_SEQ_KEY}\";  LABEL_KEY = \"{Q8_LABEL_KEY}\"\n"
    f"  LABEL_NAMES  = {q8_labels}   # the 8 DSSP states above"
)

README.md: 0.00B [00:00, ?B/s]

C:\Users\soura\code\2026\xai-starter\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\soura\.cache\huggingface\hub\datasets--lamm-mit--protein-secondary-structure-nppe2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/3.06M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/465k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1816 [00:00<?, ? examples/s]

Loaded lamm-mit/protein-secondary-structure-nppe2
  splits:   ['train', 'test']
  features: {'id': Value('int64'), 'seq': Value('string'), 'sst8': Value('string'), 'sst3': Value('string')}

  distinct Q8 labels (8): ['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T']
  label counts (first 500 seqs):   {'C': 30719, 'S': 9275, 'G': 4601, 'E': 26506, 'T': 12839, 'H': 38068, 'B': 1297, 'I': 22}

Example sequence (first 80 aa): GVGLEGGVQLSPARTRGPEFAAPEQAG
Example Q8 labels  (first 80):  CCCCCCCCSCCCCCCGGGCCCCCCCCC

To TRAIN on Q8 instead of Q3, set these in the Setup cell and re-run main():
  DATASET_NAME = "lamm-mit/protein-secondary-structure-nppe2"
  SEQUENCE_KEY = "seq";  LABEL_KEY = "sst8"
  LABEL_NAMES  = ['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T']   # the 8 DSSP states above


## Run comparison — this lesson's MLflow history

How this run compares to every prior run of the same lesson (bigger model / more epochs accumulate here as you re-run). Generated from the shared local MLflow store.

In [ ]:
# --- Run comparison -----------------------------------------------------------
# Every run of this lesson logs to the shared MLflow store. This chart shows
# how the latest run compares to all prior ones (e.g. more epochs / a bigger
# model). It accumulates as you re-run; a no-op on a fresh checkout.
import mlflow_utils as mu
mu.plot_run_comparison('plm-secondary-structure')
